### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 4: Generative AI. Exercise 3**

**Memory: when the last step is not enough**

A Markov chain only remembers the last step. Language does not work like that: the right next word often depends on words that came much earlier.
We predict the next word of short sentences about cells, first remembering one word (Markov) and then remembering more words (context), and see why remembering everything quickly becomes impossible.

**The maths**

A Markov model predicts the next word $w_t$ from the previous word only, counting how often each pair of words appears:

$$P(w_t \mid w_{t-1})$$

A model with a context of $n$ words looks at the last $n$ words:

$$P(w_t \mid w_{t-1}, w_{t-2}, \dots, w_{t-n})$$

With a vocabulary of $V$ words there are $V^n$ possible contexts, too many to count for real language. This is the problem that attention solves (next exercise).

In [ ]:
# Import libraries
import numpy as np
from collections import Counter, defaultdict

# Check installation
print("All libraries imported successfully")
print(np.__version__)

In [ ]:
# Training text: short sentences about cells
sentences = [
    "the tumour cell divides fast",
    "the normal cell divides slowly",
    "the tumour cell grows fast",
    "the normal cell grows slowly",
]
sentences = [s.split() for s in sentences]
vocabulary = sorted(set(word for s in sentences for word in s))
print("Sentences:", sentences)
print("Vocabulary:", vocabulary)
print("Vocabulary size:", len(vocabulary))

In [ ]:
# Count which word comes after each context of n words
def count_next_words(sentences, n):
    counts = defaultdict(Counter)
    for s in sentences:
        for t in range(n, len(s)):
            context = tuple(s[t - n:t])
            counts[context][s[t]] += 1
    return counts

# Probability of each next word, given a context
def next_word_probs(counts, context):
    c = counts[tuple(context)]
    total = sum(c.values())
    return {word: count / total for word, count in c.items()}

In [ ]:
# Markov: remember only the last word
counts_1 = count_next_words(sentences, n = 1)
print("After 'divides':", next_word_probs(counts_1, ["divides"]))
print("After 'cell':   ", next_word_probs(counts_1, ["cell"]))

# 'fast' or 'slowly' depends on 'tumour' or 'normal', three words before
# With only one word of memory the model cannot know: 50% each

In [ ]:
# Generate sentences one word at a time, choosing the next word at random
def generate(counts, start, n, length = 5):
    words = list(start)
    while len(words) < length:
        probs = next_word_probs(counts, words[-n:])
        words.append(np.random.choice(list(probs.keys()), p = list(probs.values())))
    return " ".join(words)

np.random.seed(0)
print("Markov model (memory of 1 word):")
for i in range(5):
    print("  ", generate(counts_1, ["the"], n = 1))
# Some sentences mix things up: "the tumour cell divides slowly"

In [ ]:
# Context: remember the last 3 words
counts_3 = count_next_words(sentences, n = 3)
print("After 'tumour cell divides':", next_word_probs(counts_3, ["tumour", "cell", "divides"]))
print("After 'normal cell divides':", next_word_probs(counts_3, ["normal", "cell", "divides"]))

np.random.seed(0)
print("\nModel with a context of 3 words:")
for i in range(5):
    print("  ", generate(counts_3, ["the", "tumour", "cell"], n = 3))

In [ ]:
# The problem: the number of possible contexts grows very fast
# A real language model has a vocabulary of about 50000 words
for V in [len(vocabulary), 50000]:
    print(f"Vocabulary of {V} words:")
    for n in [1, 2, 3, 5]:
        print(f"   last {n} words: {V**n:,} possible contexts")